# 1.0 Import libraries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

# 2.0 Data load

In [3]:
df = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_10_30_data_average_icarda_for_excel_dashboard/icarda_data.csv')

In [22]:
df.head(2)

,id_lab,functional_group,tax_name,id,genus,run,replication,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,ch4_g_digested,tddm,ch4_g_kg_dm
0,250100028.0,Legume,Medicago sativa,101045,Medicago,1,1.0,92.765482,92.765482,88.036896,201.443124,28.907088,28.907088,40.757851,70.923977,16.043934
1,250100028.0,Legume,Medicago sativa,101045,Medicago,1,2.0,92.765482,39.230801,88.036896,179.842945,24.410468,5.772120,34.385968,70.989620,13.548232


# 3.0 Average estimation

In [4]:
df.columns

Index(['id_lab', 'functional_group', 'tax_name', 'id', 'genus', 'run',
       'replication', 'dm', 'ndf', 'om', 'gas_g_dm_incubated',
       'ch4_g_dm_incubated', 'ch4_ndf', 'ch4_g_digested', 'tddm',
       'ch4_g_kg_dm'],
      dtype='object')

In [5]:
#Columns processing for nutrition complete dataframe
category_columns = ['id_lab', 'functional_group', 'tax_name', 'id', 'genus', 'run', 'replication']
numeric_columns = ['dm', 'ndf', 'om', 'gas_g_dm_incubated', 'ch4_g_dm_incubated', 'ch4_ndf', 'ch4_g_digested', 'tddm', 'ch4_g_kg_dm']

for df in [df]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [6]:
# Group by subset and id, keeping categorical columns for nutrition complete dataframe
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # # Count the number of replicates in each group
    # mean_df['n_replicates_nutrition'] = (
    #     df.groupby([ 'id_lab'])
    #       .size()
    # )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [7]:
# Creation of averaged dataframe
df2 = mean_with_replicates(df, category_columns, numeric_columns)

/tmp/ipykernel_14489/3922859182.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_14489/3922859182.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]


In [8]:
df2.head()

,id_lab,functional_group,tax_name,id,genus,run,replication,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,ch4_g_digested,tddm,ch4_g_kg_dm
0,250100028.0,Legume,Medicago sativa,101045,Medicago,1,1.0,92.77,46.88,88.04,191.27,27.20,9.58,41.44,65.73,15.09
1,250100029.0,Legume,Medicago sativa,101063,Medicago,1,1.0,94.98,50.67,89.58,194.48,29.19,5.47,54.50,53.87,16.20
2,250100030.0,Legume,Medicago sativa,101065,Medicago,1,1.0,94.56,45.96,89.34,207.87,30.60,6.29,51.99,59.32,16.98
3,250100031.0,Legume,Medicago sativa,101076,Medicago,1,1.0,94.11,50.19,89.50,199.71,28.97,5.43,52.55,55.20,14.62
4,250100032.0,Legume,Medicago sativa,101077,Medicago,1,2.0,94.82,39.17,89.20,199.79,29.25,7.08,56.12,53.88,16.23


# 4.0 Formatting

In [9]:
df2.functional_group.unique()

['Legume', 'Grass', 'CC', 'CH', 'C5', 'C6', 'grass']
Categories (7, object): ['C5', 'C6', 'CC', 'CH', 'Grass', 'Legume', 'grass']

In [10]:
df2["functional_group"] = df2["functional_group"].replace({"Grass": "Grasses", 'grass': 'Grasses', 'Legume': 'Herbaceous_legumes'})

/tmp/ipykernel_14489/2890134519.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df2["functional_group"] = df2["functional_group"].replace({"Grass": "Grasses", 'grass': 'Grasses', 'Legume': 'Herbaceous_legumes'})


In [11]:
df2.functional_group.unique()

['Herbaceous_legumes', 'Grasses', 'CC', 'CH', 'C5', 'C6']
Categories (6, object): ['C5', 'C6', 'CC', 'CH', 'Grasses', 'Herbaceous_legumes']

In [12]:
df2 = df2.rename(columns={'ch4_g_digested': 'methane_intensity'})

In [13]:
df2.head(2)

,id_lab,functional_group,tax_name,id,genus,run,replication,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,methane_intensity,tddm,ch4_g_kg_dm
0,250100028.0,Herbaceous_legumes,Medicago sativa,101045,Medicago,1,1.0,92.77,46.88,88.04,191.27,27.20,9.58,41.44,65.73,15.09
1,250100029.0,Herbaceous_legumes,Medicago sativa,101063,Medicago,1,1.0,94.98,50.67,89.58,194.48,29.19,5.47,54.50,53.87,16.20


In [14]:
df2.columns

Index(['id_lab', 'functional_group', 'tax_name', 'id', 'genus', 'run',
       'replication', 'dm', 'ndf', 'om', 'gas_g_dm_incubated',
       'ch4_g_dm_incubated', 'ch4_ndf', 'methane_intensity', 'tddm',
       'ch4_g_kg_dm'],
      dtype='object')

In [15]:
df3 = df2[['id_lab', 'id', 'tax_name', 'functional_group', 'genus', 'dm', 'ndf', 'om', 'gas_g_dm_incubated',
       'ch4_g_dm_incubated', 'ch4_ndf', 'ch4_g_kg_dm', 'methane_intensity', 'tddm']]

In [16]:
df3.head(2)

,id_lab,id,tax_name,functional_group,genus,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,ch4_g_kg_dm,methane_intensity,tddm
0,250100028.0,101045,Medicago sativa,Herbaceous_legumes,Medicago,92.77,46.88,88.04,191.27,27.20,9.58,15.09,41.44,65.73
1,250100029.0,101063,Medicago sativa,Herbaceous_legumes,Medicago,94.98,50.67,89.58,194.48,29.19,5.47,16.20,54.50,53.87


# 5.0 Data split

In [17]:
print(df3.shape)
df3.head(2)

(760, 14)


,id_lab,id,tax_name,functional_group,genus,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,ch4_g_kg_dm,methane_intensity,tddm
0,250100028.0,101045,Medicago sativa,Herbaceous_legumes,Medicago,92.77,46.88,88.04,191.27,27.20,9.58,15.09,41.44,65.73
1,250100029.0,101063,Medicago sativa,Herbaceous_legumes,Medicago,94.98,50.67,89.58,194.48,29.19,5.47,16.20,54.50,53.87


In [18]:
grasses = df3[df3['functional_group'] == 'Grasses']
legumes = df3[df3['functional_group'] == 'Herbaceous_legumes']

In [19]:
print(grasses.shape)
grasses.head(2)

(256, 14)


,id_lab,id,tax_name,functional_group,genus,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,ch4_g_kg_dm,methane_intensity,tddm
159,250100187.0,160251,Avena sterilis,Grasses,Avena,93.53,NaN,93.65,214.47,29.75,NaN,16.51,42.50,69.82
251,250200403.0,0,Hordeum vulgare,Grasses,Hordeum,94.09,48.32,85.68,212.19,30.99,6.03,17.20,36.79,84.41


In [20]:
print(legumes.shape)
legumes.head(2)

(496, 14)


,id_lab,id,tax_name,functional_group,genus,dm,ndf,om,gas_g_dm_incubated,ch4_g_dm_incubated,ch4_ndf,ch4_g_kg_dm,methane_intensity,tddm
0,250100028.0,101045,Medicago sativa,Herbaceous_legumes,Medicago,92.77,46.88,88.04,191.27,27.20,9.58,15.09,41.44,65.73
1,250100029.0,101063,Medicago sativa,Herbaceous_legumes,Medicago,94.98,50.67,89.58,194.48,29.19,5.47,16.20,54.50,53.87


# 6.0 Save data

In [21]:
df3.to_csv('/content/drive/MyDrive/lmf/output/2026_10_30_data_average_icarda_for_excel_dashboard/icarda_data_average.csv', index=False)
grasses.to_csv('/content/drive/MyDrive/lmf/output/2026_10_30_data_average_icarda_for_excel_dashboard/icarda_grasses_average.csv', index=False)
legumes.to_csv('/content/drive/MyDrive/lmf/output/2026_10_30_data_average_icarda_for_excel_dashboard/icarda_legumes_average.csv', index=False)